In [262]:
import numpy as np

In [263]:
# Positional Encoding formula
def PE(p, d_index, d):
    if d_index % 2 == 0:
        return np.sin(p / 10000 ** (d_index / d))
    else:
        return np.cos(p / (10000 ** ((d_index - 1) / d)))


def softmax(x):
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

Input

In [264]:
input_string = "lets go"
enc_input_words = np.array(["<SOS>", "let's", "to", "go"])
enc_input = np.array([[1, 0, 0, 0],
                      [0, 1, 0, 0],
                      [0, 0, 0, 1]])
enc_input

array([[1, 0, 0, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 1]])

Embedding

In [265]:
enc_emb_weights = np.array([
    [1.16, -0.77],  # <SOS>
    [-0.27, 0.82],  # let's
    [-2.19, 0.89],  # to
    [3.50, -1.74],  # go
])

enc_embeddings = enc_input @ enc_emb_weights
enc_embeddings

array([[ 1.16, -0.77],
       [-0.27,  0.82],
       [ 3.5 , -1.74]])

Position Encoding

In [266]:
enc_positional_encodings = [[PE(i_w, i_e, enc_embeddings.shape[1]) for i_e, embedding in enumerate(word)] for i_w, word in
                            enumerate(enc_embeddings)]
enc_positional_encodings

[[np.float64(0.0), np.float64(1.0)],
 [np.float64(0.8414709848078965), np.float64(0.5403023058681398)],
 [np.float64(0.9092974268256817), np.float64(-0.4161468365471424)]]

In [267]:
enc_positional_encoded = enc_embeddings + enc_positional_encodings
enc_positional_encoded

array([[ 1.16      ,  0.23      ],
       [ 0.57147098,  1.36030231],
       [ 4.40929743, -2.15614684]])

Self Attention

In [268]:
enc_W_Q = np.array([
    [2.22, 0.41],
    [0.17, -0.51],
])

enc_W_K = np.array([
    [-1.82, 0.57],
    [1.36, -0.38],
])

enc_W_V = np.array([
    [-0.43, -0.59],
    [1.33, -2.15],
])
enc_queries = enc_positional_encoded @ enc_W_Q
enc_keys = enc_positional_encoded @ enc_W_K
enc_values = enc_positional_encoded @ enc_W_V

enc_attention_scores = softmax((enc_queries @ enc_keys.T) / np.sqrt(enc_values.shape[1])) @ enc_values
enc_attention_scores

array([[ 1.54646721, -3.2416544 ],
       [ 1.48134159, -3.16442301],
       [ 1.5634693 , -3.26181755]])

Residual Connections

In [269]:
enc_residual_connected = enc_attention_scores + enc_positional_encoded
enc_residual_connected

array([[ 2.70646721, -3.0116544 ],
       [ 2.05281257, -1.8041207 ],
       [ 5.97276673, -5.41796439]])

Decoder

Decored Input

In [270]:
dec_input_words = np.array(["<SOS>", "ir", "vamos", "y", "<EOS>"])
dec_input = np.array([[1, 0, 0, 0, 0],
                      [0, 0, 0, 0, 0],
                      [0, 0, 1, 0, 0],
                      [0, 0, 0, 0, 0],
                      [0, 0, 0, 0, 0]])
dec_input

array([[1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]])

Embedding

In [271]:
dec_emb_weights = np.array([
    [-2.53, -0.97],  # <SOS>
    [1.27, 2.17],  # ir
    [0.71, 0.73],  # vamos
    [0.58, 0.69],  # y
    [-1.39, 0.76],  # <EOS>
])

dec_embeddings = dec_input @ dec_emb_weights
dec_embeddings = dec_embeddings[np.any(dec_embeddings != 0, axis=1)]
dec_embeddings

array([[-2.53, -0.97],
       [ 0.71,  0.73]])

Positional Encoding

In [272]:
dec_positional_encodings = [[PE(i_w, i_e, dec_embeddings.shape[1]) for i_e, embedding in enumerate(word)] for i_w, word in
                            enumerate(dec_embeddings)]
dec_positional_encodings

[[np.float64(0.0), np.float64(1.0)],
 [np.float64(0.8414709848078965), np.float64(0.5403023058681398)]]

In [273]:
dec_positional_encoded = dec_embeddings + dec_positional_encodings
dec_positional_encoded

array([[-2.53      ,  0.03      ],
       [ 1.55147098,  1.27030231]])

Self Attention

In [274]:
dec_W_Q = np.array([
    [-0.19, 0.24],
    [0.64, 1.47],
])

dec_W_K = np.array([
    [-0.08, 0.38],
    [1.18, 0.67],
])

dec_W_V = np.array([
    [1.26, 1.10],
    [-0.71, 0.05],
])

dec_queries = dec_positional_encoded @ dec_W_Q
dec_keys = dec_positional_encoded @ dec_W_K
dec_values = dec_positional_encoded @ dec_W_V

similarities = (dec_queries @ dec_keys.T) / np.sqrt(dec_values.shape[1])
## applying mask:
mask = np.triu(
    np.full((dec_queries.shape[0], dec_queries.shape[0]), -np.inf),
    k=1
)
similarities = similarities + mask
dec_self_attention_scores = softmax(similarities) @ dec_values
dec_self_attention_scores

array([[-3.2091    , -2.7815    ],
       [ 0.98928356,  1.70215275]])

Residual Connections

In [275]:
dec_residual_connected_1 = dec_self_attention_scores + dec_positional_encoded
dec_residual_connected_1

array([[-5.7391    , -2.7515    ],
       [ 2.54075454,  2.97245505]])

Encoder Decoder Attention

In [276]:
dec_W_Q_encoder_decoder = np.array([
    [0.90, 1.32],
    [1.00, 0.38],
])

dec_W_K_encoder_decoder = np.array([
    [0.94, 1.28],
    [-0.70, -0.97],
])

dec_W_V_encoder_decoder = np.array([
    [-1.03, 1.73],
    [1.11, -1.49],
])

dec_queries_encoder_decoder = dec_residual_connected_1 @ dec_W_Q_encoder_decoder
dec_keys_encoder_decoder = enc_residual_connected @ dec_W_K_encoder_decoder
dec_values_encoder_decoder = enc_residual_connected @ dec_W_V_encoder_decoder

dec_encoder_decoder_attention_scores = softmax((dec_queries_encoder_decoder @ dec_keys_encoder_decoder.T) / np.sqrt(
    dec_values_encoder_decoder.shape[1])) @ dec_values_encoder_decoder
dec_encoder_decoder_attention_scores

array([[ -4.11697093,   6.2395056 ],
       [-12.16589021,  18.40565338]])

Residual Connections

In [277]:
dec_residual_connected_2 = dec_encoder_decoder_attention_scores + dec_residual_connected_1
dec_residual_connected_2

array([[-9.85607093,  3.4880056 ],
       [-9.62513566, 21.37810844]])

Un-embedding (fully connected)

In [278]:
un_emb_weights = np.array([
    [0.44, -0.14, -1.73, 0.67, 0.30],
    [-0.37, -1.69, -0.53, -0.39, 1.80],
])

un_emb_biases = np.array([-1.01, 0.21, 0.44, -1.42, 0.15])

un_embeddings = dec_residual_connected_2 @ un_emb_weights + un_emb_biases
un_embeddings

array([[ -6.63723328,  -4.30487954,  15.64235974,  -9.38388971,
          3.4715888 ],
       [-13.15495981, -34.57148426,   5.76108722, -16.20630318,
         35.74305449]])

In [279]:
soft_maxed_un_embeddings = softmax(un_embeddings)
soft_maxed_un_embeddings

array([[2.10908187e-10, 2.17281024e-09, 9.99994818e-01, 1.35280658e-11,
        5.17963420e-06],
       [5.80580191e-22, 2.90257749e-31, 9.52789716e-14, 2.74587413e-23,
        1.00000000e+00]])

Output

In [280]:
args = np.argmax(soft_maxed_un_embeddings, axis=1)[:-1]
output_string = " ".join(dec_input_words[args])
dec_input_words[args]

array(['vamos'], dtype='<U5')

In [281]:
print(f"{input_string} => {output_string}")

lets go => vamos
